# 04. Переход bronze → silver

Источник — parquet из `data/bronze`, результат — `data/silver`. Вся логика лежит в `scripts/silver_build.py`;
тетрадка только вызывает функции и показывает проверки, поэтому повторный запуск воспроизводит тот же silver.

Решения по очистке — в `docs_favorita/0_raw_data_exploration/cleaning_decisions.md`. Кратко:

| Таблица silver | Что делается | Решения |
|---|---|---|
| `stores` | без изменений | — |
| `items` | + `is_weighted` | 4.3 |
| `oil` | полный календарь, заполнение из прошлого, `is_imputed` | 3.2 |
| `transactions` | + 118 пар без чеков, `transactions_missing` | 3.3 |
| `holiday_features` | признаки праздников на сетке (date, store_nbr) | 1.2, 2.4, S2 |
| `sales_history` | все строки train с флагами, без восстановления нулей (для EDA) | C1–C4 |
| `sales_daily` | плотная сетка с нулями с 2015-06-01, файл на месяц | C1–C4, 3.4, S3 |

## Подготовка

In [1]:
import os
import sys
from pathlib import Path

# Пути в scripts относительны корня проекта. В VS Code корень задан jupyter.notebookFileRoot,
# при запуске из командной строки (jupyter nbconvert) рабочий каталог — notebooks/
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import duckdb
import pandas as pd

from scripts import silver_build as sb

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

Соединение in-memory, как в `01_parquet_convertation`: файл базы не нужен. Ограничение памяти и каталог
для сброса на диск нужны для `sales_history` (сортировка 125 млн строк) и проверок `sales_daily`.

In [2]:
con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")  # оставить память Windows и Jupyter
con.execute("SET temp_directory = 'data/tmp'")  # куда сбрасывать при нехватке RAM
con.execute("SET preserve_insertion_order = false")  # ORDER BY при этом соблюдается

sb.register_bronze(con)

Параметры, которые определяют результат. Их смысл — в docstring модуля и в журнале решений.

In [3]:
pd.DataFrame(
    {
        "value": [
            sb.WINDOW_START,
            sb.TRAIN_END,
            sb.TEST_END,
            sb.OUTLIER_Z,
            sb.OUTLIER_MIN_SALES,
            sb.OUTLIER_MIN_ROWS,
            sb.EARTHQUAKE_SPIKE,
            sb.CHRISTMAS_SPIKE_DAYS,
            sb.WEIGHTED_SHARE,
        ]
    },
    index=[
        "WINDOW_START",
        "TRAIN_END",
        "TEST_END",
        "OUTLIER_Z",
        "OUTLIER_MIN_SALES",
        "OUTLIER_MIN_ROWS",
        "EARTHQUAKE_SPIKE",
        "CHRISTMAS_SPIKE_DAYS",
        "WEIGHTED_SHARE",
    ],
)

,value
WINDOW_START,2015-06-01
TRAIN_END,2017-08-15
TEST_END,2017-08-31
OUTLIER_Z,5.0
OUTLIER_MIN_SALES,100
OUTLIER_MIN_ROWS,30
EARTHQUAKE_SPIKE,"(2016-04-18, 2016-04-21)"
CHRISTMAS_SPIKE_DAYS,"(19, 24)"
WEIGHTED_SHARE,0.5


## 1. Справочники и внешние ряды

Маленькие таблицы — секунды. Сразу проверяются: число строк, ключ, ожидаемые числа из профилирования
(293 весовых товара, 118 пропусков в transactions).

In [4]:
small = ["stores", "items", "oil", "transactions"]
sb.build_all(con, small)
pd.concat([r.to_frame() for r in sb.verify_all(con, small)], ignore_index=True)

,table,check,expected,actual,passed
0,stores,rows,54,54,True
1,stores,duplicates.store_nbr,0,0,True
2,items,rows,4100,4100,True
3,items,duplicates.item_nbr,0,0,True
4,items,is_weighted,293,293,True
5,oil,rows = календарные дни,1704,1704,True
6,oil,duplicates.date,0,0,True
7,oil,nulls после первого значения,0,0,True
8,oil,исходные значения не изменены,0,0,True
9,oil,непустые значения = bronze,1175,1175,True


In [5]:
con.sql("SELECT * FROM 'data/silver/oil.parquet' ORDER BY date LIMIT 7").df()

,date,dcoilwtico,dcoilwtico_raw,is_imputed
0,2013-01-01,NaN,NaN,False
1,2013-01-02,93.14,93.14,False
2,2013-01-03,92.97,92.97,False
3,2013-01-04,93.12,93.12,False
4,2013-01-05,93.12,NaN,True
5,2013-01-06,93.12,NaN,True
6,2013-01-07,93.20,93.20,False


## 2. Признаки праздников

Сетка (date, store_nbr) от начала train до конца test: 1704 дня × 54 магазина. Расстояния до праздников
считаются по всему диапазону `holidays_events` (2012-03 … 2017-12), поэтому на краях сетки они тоже известны.

In [6]:
sb.build_holiday_features(con)
sb.verify_all(con, ["holiday_features"])[0].to_frame()

,table,check,expected,actual,passed
0,holiday_features,rows = дни × магазины,92016,92016,True
1,holiday_features,"duplicates.(date, store_nbr)",0,0,True
2,holiday_features,nulls в расстояниях до праздников,0,0,True
3,holiday_features,национальные праздники во всех магазинах,0,0,True


In [7]:
con.sql("""
SELECT *
FROM 'data/silver/holiday_features.parquet'
WHERE store_nbr = 1 AND date BETWEEN '2016-04-28' AND '2016-05-10'
ORDER BY date
""").df()

,date,store_nbr,is_holiday,is_transferred_from,is_bridge,is_work_day,is_event,holidays_count,holiday_locale,days_since_prev_holiday,days_to_next_holiday
0,2016-04-28,1,False,False,False,False,True,0,NaN,34,3
1,2016-04-29,1,False,False,False,False,True,0,NaN,35,2
2,2016-04-30,1,False,False,False,False,True,0,NaN,36,1
3,2016-05-01,1,True,False,False,False,True,1,National,0,0
4,2016-05-02,1,False,False,False,False,True,0,NaN,1,5
5,2016-05-03,1,False,False,False,False,True,0,NaN,2,4
6,2016-05-04,1,False,False,False,False,True,0,NaN,3,3
7,2016-05-05,1,False,False,False,False,True,0,NaN,4,2
8,2016-05-06,1,False,False,False,False,True,0,NaN,5,1
9,2016-05-07,1,True,False,False,False,True,1,National,0,0


## 3. История продаж (разреженная)

Все строки train, ничего не добавляется и не удаляется. Самый долгий шаг вместе с `sales_daily`:
сортировка 125 млн строк со сбросом на диск в `data/tmp`.

In [8]:
sb.build_sales_history(con)

WindowsPath('data/silver/sales_history.parquet')

## 4. Плотная таблица продаж

Сетка пар (магазин, товар) × даты от `max(первая продажа, 2015-06-01)` до 2017-08-15. Пишется по месяцам
во временный каталог, который подменяет `data/silver/sales_daily/` только после успеха.

In [9]:
sb.build_sales_daily(con)

2015-06 записан


2015-07 записан


2015-08 записан


2015-09 записан


2015-10 записан


...

2017-05 записан


2017-06 записан


2017-07 записан


2017-08 записан


WindowsPath('data/silver/sales_daily')

## 5. Проверки

Инварианты из журнала решений. Для `sales_daily`: размер сетки, уникальность ключа, точная сумма
`unit_sales_raw` (DECIMAL, не зависит от порядка сложения) равна bronze в окне, число исходных строк равно bronze
в окне, нет 25.12 и дат раньше окна, в закрытые дни нет продаж, у восстановленных строк нет onpromotion.

In [10]:
reports = sb.verify_all(con)
checks = pd.concat([r.to_frame() for r in reports], ignore_index=True)
checks

,table,check,expected,actual,passed
0,stores,rows,54,54,True
1,stores,duplicates.store_nbr,0,0,True
2,items,rows,4100,4100,True
3,items,duplicates.item_nbr,0,0,True
4,items,is_weighted,293,293,True
5,oil,rows = календарные дни,1704,1704,True
6,oil,duplicates.date,0,0,True
7,oil,nulls после первого значения,0,0,True
8,oil,исходные значения не изменены,0,0,True
9,oil,непустые значения = bronze,1175,1175,True


In [11]:
for r in reports:
    r.raise_if_failed()
print("Все проверки пройдены")

Все проверки пройдены


## 6. Что получилось

In [12]:
sb.register_silver(con)
con.sql("""
SELECT
    count(*)                                          AS n_rows,
    count(*) FILTER (WHERE is_restored)               AS n_restored,
    count(*) FILTER (WHERE is_store_closed)           AS n_store_closed,
    count(*) FILTER (WHERE is_return)                 AS n_return,
    count(*) FILTER (WHERE is_outlier)                AS n_outlier,
    count(*) FILTER (WHERE is_event_spike)            AS n_event_spike,
    round(100 * avg(CAST(is_restored AS INTEGER)), 1) AS pct_restored
FROM silver_sales_daily
""").df()

,n_rows,n_restored,n_store_closed,n_return,n_outlier,n_event_spike,pct_restored
0,119312458,40889984,823471,5015,9642,2378461,34.3


In [13]:
con.sql("""
SELECT
    count(*)                                 AS n_rows,
    count(*) FILTER (WHERE is_outlier)       AS n_outlier,
    count(*) FILTER (WHERE is_event_spike)   AS n_event_spike,
    count(*) FILTER (WHERE is_outlier AND is_event_spike) AS n_outlier_on_event,
    count(*) FILTER (WHERE is_reliable_period) AS n_reliable
FROM silver_sales_history
""").df()

,n_rows,n_outlier,n_event_spike,n_outlier_on_event,n_reliable
0,125497040,12636,2404340,1377,77599003


In [14]:
# Размер файлов silver
files = sorted(p for p in Path("data/silver").rglob("*.parquet"))
pd.DataFrame(
    {
        "file": [p.as_posix() for p in files],
        "mb": [round(p.stat().st_size / 2**20, 1) for p in files],
    }
)

,file,mb
0,data/silver/holiday_features.parquet,0.0
1,data/silver/items.parquet,0.0
2,data/silver/oil.parquet,0.0
3,data/silver/sales_daily/2015-06.parquet,9.8
4,data/silver/sales_daily/2015-07.parquet,10.4
5,data/silver/sales_daily/2015-08.parquet,10.7
6,data/silver/sales_daily/2015-09.parquet,10.8
7,data/silver/sales_daily/2015-10.parquet,11.7
8,data/silver/sales_daily/2015-11.parquet,11.6
9,data/silver/sales_daily/2015-12.parquet,12.0
